# Ghép lóe (Wu et al. + FlareX) lên ảnh drone VisDrone và ảnh nền Flickr24K

Notebook này **không train**. Nó ghép lớp lóe của Wu et al. (ICCV 2021) và, nếu gắn, của **FlareX** (NeurIPS 2025, có
sẵn vòng quầng và bóng ma tròn/lục giác rõ nét) lên ảnh sạch: ảnh giao thông ban ngày của VisDrone và **ảnh nền Flickr24K** của Flare7K (23 949 ảnh Flickr
của Zhang et al., CVPR 2018, đã lọc ảnh có lóe và cháy sáng; cùng nguồn ảnh nền với bài Wu). **Lớp lóe chia đúng theo `FLARE_SHARES`:** 60% Wu `captured/` (chụp thật: bóng ma, halo, màn chói), 20%
Wu `simulated/` (tia sao, quầng chói), 20% FlareX; không gắn FlareX thì phần đó chia lại theo tỉ lệ của hai phần Wu
(75 / 25). Mặc định 5 000 cặp **640×640**, 80%
trên ảnh Flickr (người, phố, phong cảnh, trong nhà, ngày và chiều tối) và 20% trên VisDrone. **Ít trùng với bộ cũ**
`visdrone-lens-flare-640`: ảnh VisDrone mà bộ cũ đã dùng được xếp sau cùng (`VISDRONE_AVOID_USED`), nên khi gắn bản
VisDrone gốc (3 320 ảnh qua lọc, bộ cũ dùng 2 000) mọi cặp VisDrone là ảnh chưa dùng; chỉ chung chuỗi (cùng nơi chụp).
**Phải gắn `visdrone-lens-flare-640`** (bộ cũ) để Cell 4 biết ảnh nào đã dùng: lần dựng đầu (24/10/2026) không gắn nên
633/1 000 cặp VisDrone trùng ảnh nền bộ cũ; giờ thiếu bộ cũ thì Cell 4 dừng (hoặc đặt `VISDRONE_AVOID_USED = False`). Lớp Wu ghép theo
**cách của bài Wu** (`flare_removal/python/synthesis.py:add_flare` của Google, viết lại bằng numpy + OpenCV), xuất
ảnh xem trước kiểu `01_wu_test_that.png` (ảnh có lóe → ảnh sạch) để chỉnh hệ số sáng, rồi tạo Kaggle dataset cặp ảnh.

**Cách ghép một cặp** (mọi số ở Cell 1). Ba chỗ khác Wu, đều chỉnh được: hệ số sáng lớp lóe mặc định chung cho 3 kênh
(3–10, lệch màu ±20%) thay vì từng kênh 0–10, hệ số sáng cảnh 0,6–1,2 thay vì 0–1,2 (đo ở Cell 5: gần ảnh thật của Wu
hơn), lớp lóe mờ dần ở mép phần ảnh thật (`FEATHER`), vì bản 640×640 có dải đệm mà ảnh gốc của Wu không có, và
**bóng ma, vòng quầng được làm sáng lên** (`GHOST_BOOST`, chỉ ảnh `captured/`; tách riêng phần bóng ma khỏi tia sao và
quầng chói quanh mặt trời rồi chỉ nhân phần đó): chúng có sẵn trong ảnh chụp của Wu nhưng
tối hơn lõi 10–40 lần, ghép thẳng thì chỉ thấy một cục lóe.

1. Gamma ngẫu nhiên `γ ∈ [1,8; 2,2]`. Lớp lóe và ảnh sạch đều đưa về ánh sáng tuyến tính bằng `x^γ`.
2. Lớp lóe: bỏ nền DC (`remove_background`), biến hình ngẫu nhiên quanh tâm (xoay ±180°, dịch N(0, 10 px), nghiêng
   ±20°, co giãn 0,9–1,2 × 1,25 cho ô 640 × `WU_SIZE` 0,85), cắt giữa `OUT_SIZE`, lật ngẫu nhiên, cân bằng trắng rồi nhân **hệ số sáng từng kênh**
   (`FLARE_GAIN_MODE`, `FLARE_MIN_GAIN`–`FLARE_MAX_GAIN`), làm mờ Gauss σ ∈ [0,1; 3], cộng offset DC ∈ [−0,02; 0,02].
3. Ảnh sạch: thu cạnh ngắn về `SCENE_SHORT`, cắt ngẫu nhiên `OUT_SIZE`, lật, nhiễu Gauss, nhân **hệ số sáng cảnh**
   `U[SCENE_GAIN]`.
4. Ảnh có lóe = ảnh sạch + lớp lóe (tuyến tính), đưa về sRGB, cắt về [0, 1].
5. Ảnh đích (`gt`): ảnh sạch. **Mặc định `KEEP_LIGHT = False`, như Wu: ảnh đích không còn lóe lẫn nguồn sáng** (người
   dùng 24/10/2026 muốn xoá sạch, không để vệt trắng). Với `KEEP_LIGHT = True` (như repo), ảnh đích giữ lại **lõi nguồn sáng** (mặt trời), tức
   phần gần bão hoà của lớp lóe **đo trước khi nhân hệ số sáng**, trọng số tăng tuyến tính từ 225 tới 252 như
   `corruptions/lens_flare.py:wu_light`; cặp FlareX thì ảnh đích = cảnh + ảnh nguồn sáng `gt/` của FlareX.

| Input (*Add Input*) | bắt buộc? | dùng để |
|---|---|---|
| `buidinhkhoi/wu-lens-flare-640` | có | lớp lóe (`captured/` 2 001, `simulated/` 3 000, 640×640) |
| (không cần gắn) | | **ảnh nền Flickr24K**: Cell 4b tự tải `Flickr24K.zip` từ link Google Drive trong README của `ykdai/Flare7K` (`BACKGROUND_DRIVE_ID`), chỉ giải nén `BACKGROUND_CANDIDATES` ảnh rút ngẫu nhiên rồi lọc (bỏ ảnh nhỏ, tối, đen trắng, cháy sáng và 20% ảnh mờ nhất). Đã có bản trên Kaggle thì gắn làm Input (thư mục `Flickr24K/` hay file zip), Cell 4b đọc thẳng |
| `kushagrapandya/visdrone-dataset` (hoặc `banuprasadb/visdrone-dataset`) | không (nên gắn) | ảnh VisDrone gốc (nhãn gốc `annotations/` hoặc YOLO `labels/`); Cell 4 tự lọc ảnh ban ngày có xe, cùng ngưỡng với `visdrone-day-builder` (~3 320 ảnh). Gắn cả bản gốc lẫn `visdrone-day-2000` thì Cell 4 lấy ảnh từ bản gốc |
| `<tài khoản>/visdrone-lens-flare-640` (bộ cũ) và/hoặc `<tài khoản>/visdrone-day-2000` | không (nên gắn) | chỉ để biết ảnh VisDrone nào bộ cũ đã dùng (cột `scene` / `file` của `index.csv`); không đọc ảnh của chúng. Chỉ gắn `visdrone-day-2000` mà không có bản gốc thì ảnh lấy từ đó, cặp VisDrone nào cũng trùng cảnh với bộ cũ |
| `lishenqu/flarex` | không (nên gắn) | FlareX: Cell 4 tìm `Flare2D/input` (9 500 lớp lóe 1440×1440) và `Flare2D/gt` (nguồn sáng tương ứng), đọc thẳng từ Input, **không tải** 49,7 GB. `FLARE_SHARES['flarex']` (0,2) cặp dùng lớp FlareX, ghép như `flare7kpp_dataset.py` của FlareX; ảnh đích = cảnh + ảnh nguồn sáng của FlareX |
| `thomasdanielardhali/how-to-train-nn-for-flare-removal-dataset` | không | tập test của Wu (20 cặp thật, 38 cặp tổng hợp): đặt cạnh ảnh của mình để so, và đo cùng chỉ số |

## Chỉnh hệ số sáng
1. Chạy Cell 1–5. Cell 5 ghi `/kaggle/working/preview_ghep.png` (20 cặp, như `01_wu_test_that.png`) và
   `preview_wu_that.png` (20 cặp thật của Wu), rồi in bảng **độ lóe** của ảnh mình và của Wu.
2. Lóe quá nhạt/quá gắt thì sửa ở Cell 1 rồi chạy lại Cell 1 và Cell 5 (chỉ vài giây): lớp Wu `FLARE_MIN_GAIN`,
   `FLARE_MAX_GAIN`, `GHOST_BOOST`; lớp FlareX `FLAREX_BRIGHTNESS`, `FLAREX_SIDE` (nhỏ hơn: thấy trọn chuỗi bóng ma; lớn
   hơn: lóe to hơn khung); cảnh `SCENE_GAIN`. Bảng độ lóe (tách riêng Wu và FlareX) cho biết đang lệch về phía nào.
3. Ưng rồi thì chạy Cell 6–9 để tạo dataset (hoặc *Save Version → Save & Run All*). 5 000 cặp: ghép khoảng
   15–25 phút trên 4 CPU, dataset khoảng 6–8 GB, upload thêm 20–40 phút.

**Dùng khi train:** dataset mới tên `scene-lens-flare-640`, cùng bố cục với `visdrone-lens-flare-640` (`index.csv`,
`gt/`, `input/`), nên notebook train đọc như cũ (`data.scene_pairs_*`). Gắn **một trong hai** làm Input, không gắn cả
hai (`find_scene_pairs_root` từ chối khi thấy hai bộ).

## Chạy
*Settings*: **Accelerator: None**, **Internet: On** (tải Flickr24K, cài `gdown` và `kaggle` CLI). Secret **`KAGGLE_API_TOKEN`** (bật
*Attached*) để tự tạo dataset; không có thì Cell 8 đóng gói vào `/kaggle/working` để tạo tay từ *Output → New Dataset*.

## Cell 1 — Lựa chọn

In [ ]:
from pathlib import Path

# --- ghép (mặc định = synthesis.py của Wu et al.)
OUT_SIZE = 640                # cạnh ảnh vuông đầu ra, như các bộ 640×640 khác của repo (Wu: training_res 512);
                              # Cell 6 kiểm từng cặp đúng OUT_SIZE×OUT_SIZE
SCENE_SHORT = 640             # thu cạnh ngắn ảnh drone về đây rồi cắt ngẫu nhiên OUT_SIZE; ảnh có cạnh ngắn nhỏ hơn
                              # OUT_SIZE (VisDrone 960×540, ~5%) bị bỏ ở Cell 4, không phóng to
GAMMA = (1.8, 2.2)
# Mặc định đã chỉnh cho gần 20 cặp ảnh thật của Wu (đo 21/10/2026 trên 60 cặp, xem bảng ở Cell 5):
#   Wu gốc ('per_channel', 0–10, cảnh 0–1,2): lóe TB 0,039, vùng lóe 0,073; lóe lệch màu xanh/tím, cảnh nhiều khi tối đen
#   mặc định ('shared', 3–10, cảnh 0,6–1,2):   lóe TB 0,052, vùng lóe 0,120; lóe trắng/ấm như ảnh thật
#   ảnh thật của Wu:                            lóe TB 0,060, vùng lóe 0,167
FLARE_MAX_GAIN = 10.0         # hệ số sáng lớn nhất của lớp lóe trên ánh sáng tuyến tính (Wu: 10)
FLARE_GAIN_MODE = 'shared'    # 'shared': một hệ số U[FLARE_MIN_GAIN, max] cho cả 3 kênh × lệch màu U[0,8; 1,2]
                              # 'per_channel': mỗi kênh U[0, max] độc lập (đúng Wu; lóe hay lệch màu mạnh)
FLARE_MIN_GAIN = 3.0          # chỉ dùng với 'shared'
SCENE_GAIN = (0.6, 1.2)       # hệ số sáng ảnh sạch (Wu: (0,0, 1,2), làm nhiều ảnh ban ngày tối đen)
NOISE = 0.01                  # độ lệch chuẩn nhiễu Gauss (Wu train.py: 0,01)
BLUR_SIGMA = (0.1, 3.0)
DC_OFFSET = 0.02
ROTATION = 180.0              # độ
SHIFT_STD = 10.0              # px
SHEAR = 20.0                  # độ (Wu: π/9)
WU_SIZE = 0.85                # cỡ lớp lóe Wu so với bản chỉnh 21/10/2026: người dùng (24/10/2026) thấy lóe Wu hơi to,
                              # muốn nhỏ đi 10-20%. 1 = như cũ. FlareX không đổi (FLAREX_SCALE, FLAREX_SIDE)
SCALE = tuple(WU_SIZE * s for s in (1.125, 1.5))  # co giãn lớp Wu. Wu: (0,9, 1,2) với ô cắt 512; ở OUT_SIZE 640 nhân thêm
                              # 640/512 = 1,25 để lóe chiếm khung như ở 512 (ô 512 chỉ lấy 80% ảnh lóe 640), rồi × WU_SIZE
# Tỉ lệ lớp lóe của dataset, chia ĐÚNG số cặp trong mỗi nguồn ảnh nền và mỗi split (Cell 6), không rút ngẫu nhiên.
# Người dùng (24/10/2026): captured 60% vì ảnh ban ngày hay có lóe trắng; giữ simulated (tia sao) và FlareX mỗi
# phần 20%. Không gắn FlareX: phần của nó chia lại theo tỉ lệ các phần còn lại (captured 75%, simulated 25%).
FLARE_SHARES = {'captured': 0.6, 'simulated': 0.2, 'flarex': 0.2}
KEEP_LIGHT = False            # False = như Wu: ảnh đích không còn lóe LẪN nguồn sáng (người dùng 24/10/2026: "xoá sạch",
                              # không để vệt trắng). True = như repo (đích giữ lõi nguồn sáng, cặp FlareX giữ ảnh gt/ của nó)
LIGHT_LOW, LIGHT_HIGH = 225.0, 252.0
FEATHER = 32                  # px: lớp lóe mờ dần về 0 ở mép phần ảnh thật (dải 640×477 trong ảnh đệm 640×640)
# Hiện bóng ma và vòng quầng: ảnh captured/ có chúng nhưng tối hơn lõi 10–40 lần, nên ghép lên ảnh ban ngày chỉ còn
# thấy một cục lóe. Tách phần bóng ma ra rồi chỉ nhân phần đó (xem reveal_ghosts ở Cell 2):
#   quầng chói G(r) = trung vị theo góc của mức sáng ở mỗi bán kính r quanh lõi (quầng gần như tròn đều),
#   phần "khối" B = mở hình thái (opening) đĩa GHOST_OPEN px: xoá tia sao mảnh, giữ đĩa/vòng bóng ma,
#   bóng ma = max(B − G, 0) (làm mịn σ 2 px); lớp lóe += (GHOST_BOOST − 1) · bóng ma, cùng màu với chỗ đó.
# Nhờ vậy tia sao và quầng quanh mặt trời giữ nguyên cỡ (thử nhân đều mọi vùng tối thì quầng phình thành mảng trắng).
GHOST_BOOST = 12.0            # 1 = tắt (như Wu). Đo 21/10/2026 trên ảnh captured (60 cặp): tắt → lóe TB 0,075, vùng
                              # lóe 0,161; 12 → 0,076 và 0,162 (ảnh thật của Wu 0,060 và 0,167): bóng ma hiện ra mà
                              # tổng lượng lóe gần như không đổi, vì nền mờ bị trừ đi
GHOST_OPEN = 7                # px (ở 640): tia sao mảnh hơn cỡ này bị bỏ khỏi phần bóng ma
GHOST_CAP = 0.15              # bóng ma sau khi nhân không sáng quá mức này (tuyến tính, trước hệ số sáng lớp lóe);
                              # bóng ma vốn đã sáng hơn thì giữ nguyên, không nhân thêm
GHOST_PARTS = ('captured',)   # simulated/ chỉ có tia sao, không có bóng ma

# --- FlareX (Qu et al., NeurIPS 2025; Kaggle lishenqu/flarex, phần Flare2D): 9 500 lớp lóe 1440×1440 có sẵn vòng quầng,
# bóng ma tròn/lục giác, vệt ngang, kèm ảnh nguồn sáng riêng (gt/). Ghép theo flare7kpp_dataset.py của FlareX.
FLAREX_SIDE = 1280            # thu ảnh 1440 về cạnh này (BOX trên ánh sáng tuyến tính) trước khi biến hình và cắt giữa
                              # OUT_SIZE. 1280 với ô 640 = 1024 với ô 512 (lóe chiếm khung như lúc chỉnh). FlareX gốc
                              # không thu (1440): lóe to, chỉ thấy phần giữa; nhỏ hơn: thấy trọn cả chuỗi bóng ma
FLAREX_BRIGHTNESS = (0.8, 3.0)  # nhân độ sáng lớp lóe (ColorJitter brightness của FlareX)
FLAREX_MAX_MEAN = 0.10        # trần lượng lóe trung bình trên khung (tuyến tính) của một lớp FlareX: vài loại của FlareX
                              # (vd 7250, 9050) là màn chói phủ kín khung, ×3 thì trắng xoá; lớp vượt trần được giảm
                              # đều cho bằng trần. None = không giới hạn (như FlareX)
FLAREX_SCALE = (0.8, 1.5)     # co giãn (FlareX); xoay ±ROTATION, nghiêng ±SHEAR như Wu
FLAREX_TRANSLATE = 300 / 1440  # dịch ngẫu nhiên tới ± tỉ lệ này của cạnh (FlareX: 300 px trên 1440)

# --- ảnh nền Flickr24K (Flare7K: 23 949 ảnh Flickr của Zhang et al., CVPR 2018, Flare7K đã lọc ảnh có lóe và cháy
# sáng; Wu et al. dùng cùng nguồn ảnh này làm cảnh)
BACKGROUND_FRACTION = 0.8     # tỉ lệ cặp dùng ảnh Flickr (còn lại VisDrone: 0,2 × 5 000 = 1 000 cặp, vừa số ảnh VisDrone
                              # mà bộ cũ chưa dùng); không gắn VisDrone thì mọi cặp dùng Flickr; 0 = chỉ VisDrone
VISDRONE_AVOID_USED = True    # ảnh VisDrone đã có trong bộ cũ (visdrone-lens-flare-640 / visdrone-day-2000) chỉ dùng khi
                              # đã hết ảnh chưa dùng; Cell 6 in số cặp còn trùng. Bật mà không gắn bộ cũ thì Cell 4 dừng
BACKGROUND_DRIVE_ID = '1GNFGWfUbgXfELx5fZtjTjU2qqWnEa-Lr'  # Flickr24K.zip: link Google Drive "Background Images" trong
                              # README của github.com/ykdai/Flare7K. Drive báo hết lượt tải thì tạo bản sao vào Drive của
                              # bạn, chia sẻ "bất kỳ ai có đường liên kết", dán ID bản sao vào đây
BACKGROUND_ZIP = Path('/tmp/flickr24k.zip')
BACKGROUND_DIR = Path('/tmp/flickr24k')  # chỉ giải nén các ảnh ứng viên vào đây, không giải nén cả zip
BACKGROUND_CANDIDATES = 10000  # số ảnh rút ngẫu nhiên để lọc; cần ~N_PAIRS · BACKGROUND_FRACTION = 4 000 ảnh qua lọc
BACKGROUND_MIN_SHORT = 400    # px: cạnh ngắn nhỏ hơn thì bỏ. 400–640 được phóng lên 640 (LANCZOS); repo thu cặp về cạnh
                              # ngắn 384 khi train (data.scene_pairs_short_side), nên chi tiết của ảnh gốc vẫn còn
BACKGROUND_MIN_LUMA = 50      # độ sáng trung bình tối thiểu (0–255): ảnh đêm làm đích tối, ngược với việc làm sáng
BACKGROUND_MIN_CHROMA = 4.0   # độ lệch màu trung bình tối thiểu (|R−G| + |G−B|, 0–255): bỏ ảnh đen trắng
BACKGROUND_MAX_SATURATED = 0.02  # tỉ lệ pixel cháy trắng (luma > 250) tối đa, như VisDrone
BACKGROUND_DROP_BLURRY = 0.2  # bỏ phần ảnh mờ nhất trong số ảnh qua lọc (phương sai Laplace ở cạnh ngắn 384, thấp nhất):
                              # ảnh đích phải nét, mà ảnh Flickr hay có hậu cảnh xoá phông
DOWNLOAD_RETRIES, RETRY_WAIT = 3, 600  # Google Drive hết lượt tải: thử lại sau 10 phút, tối đa 3 lần

# --- xem trước, dataset
SEED = 0
N_PREVIEW = 20                # số cặp trong preview_ghep.png
N_STATS = 200                 # số cặp dùng để đo độ lóe
N_PAIRS = 5000                # số cặp trong dataset (mỗi ảnh sạch dùng ~1 lần)
VAL_EVERY = 10                # nhóm lóe thứ 0, 10, 20..., ảnh test-dev của VisDrone và ảnh Flickr thứ 0, 10, 20... làm valid
BUILD_DATASET = True          # False: chỉ xem trước

SLUG = f'scene-lens-flare-{OUT_SIZE}'  # bản cũ (chỉ VisDrone, 2 000 cặp) là visdrone-lens-flare-640: giữ nguyên
TITLE = f'Lens flare on Flickr and VisDrone, {OUT_SIZE}x{OUT_SIZE}'
SUBTITLE = f'Flickr and drone photos with Wu et al. / FlareX lens flare, paired'
PUBLIC = False
KAGGLE_OWNER = None

UPLOAD = Path('/tmp/upload/wu_visdrone')
OUTPUT_DIR = Path('/kaggle/working')
REPORT = OUTPUT_DIR / 'wu_visdrone_report.md'
INPUT = Path('/kaggle/input')

## Cell 2 — Hàm dùng chung (chỉ định nghĩa, chưa chạy gì)

In [ ]:
import csv
import json
import math
import os
import posixpath
import re
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor

import cv2
import numpy as np
from PIL import Image

cv2.setNumThreads(1)
IMAGE_SUFFIXES = {'.png', '.jpg', '.jpeg'}


def gb(n):
    return f'{n / 1e9:.2f} GB'


def run(command, env=None, timeout=None):
    result = subprocess.run(command, env=env, capture_output=True, text=True, timeout=timeout)
    return result.returncode, result.stdout, result.stderr


def kaggle(*args, env, timeout=None):
    code, out, err = run([shutil.which('kaggle') or 'kaggle', *args], env=env, timeout=timeout)
    return code, (out + err).strip()


def glob_depth(base, name, depth=6):
    """Mọi đường dẫn ``base/*/.../name`` tới ``depth`` tầng, nông trước."""
    for level in range(depth):
        yield from sorted(Path(base).glob('/'.join(['*'] * level + [name])))


def find_wu_flares(base):
    """Thư mục có captured/ (≥ 2 001 PNG) và simulated/ (≥ 3 000 PNG). Gắn cả bản gốc 1008×752 (vd ipythonx/flare-image)
    lẫn wu-lens-flare-640 thì lấy bản OUT_SIZE×OUT_SIZE: cách ghép (dải đệm, FEATHER, SCALE) chỉnh cho bản đó."""
    found = []
    for captured in glob_depth(base, 'captured'):
        root = captured.parent
        pngs = sorted(captured.glob('*.png'))
        if (root / 'simulated').is_dir() and len(pngs) >= 2001 and len(list((root / 'simulated').glob('*.png'))) >= 3000:
            with Image.open(pngs[0]) as image:
                found.append((image.size != (OUT_SIZE, OUT_SIZE), root, image.size))
    if len(found) > 1:
        print('lớp lóe Wu gắn nhiều bản:', ', '.join(f'{root} ({w}×{h})' for _, root, (w, h) in found))
    if found and all(other for other, _, _ in found):
        print(f'CẢNH BÁO: không có bản {OUT_SIZE}×{OUT_SIZE} (buidinhkhoi/wu-lens-flare-640); cách ghép chỉnh cho bản đó')
    return min(found, key=lambda item: item[0])[1] if found else None  # bản OUT_SIZE trước, rồi theo thứ tự tìm


def find_visdrone_day(base):
    """Thư mục của visdrone-day-builder: index.csv + images/."""
    for index in glob_depth(base, 'index.csv'):
        if (index.parent / 'images').is_dir():
            with open(index) as handle:
                if 'source_split' in (handle.readline()):
                    return index.parent
    return None


def find_used_scenes(base):
    """{tên ảnh VisDrone đã dùng}: cột ``scene`` của bộ cặp cũ (index.csv + gt/ + input/, vd visdrone-lens-flare-640) và
    cột ``file`` của visdrone-day-2000 (bộ cũ ghép từ đúng 2 000 ảnh đó), cùng danh sách thư mục đã đọc."""
    used, roots = set(), []
    for index in glob_depth(base, 'index.csv'):
        folder = index.parent
        with open(index, newline='') as handle:
            reader = csv.DictReader(handle)
            columns = reader.fieldnames or []
            if {'scene', 'scene_split', 'flare'} <= set(columns) and (folder / 'gt').is_dir():
                used |= {row['scene'] for row in reader if row['scene_split'] != 'flickr24k'}
            elif 'source_split' in columns and (folder / 'images').is_dir():
                used |= {row['file'] for row in reader}
            else:
                continue
        roots.append(str(folder))
    return used, roots


def find_visdrone_raw(base):
    """{'train': thư mục chứa images/ và annotations/, ...} của VisDrone2019-DET gốc."""
    found = {}
    for images in glob_depth(base, 'images'):
        if images.is_dir() and images.parent.name.startswith('VisDrone2019-DET-'):
            found.setdefault(images.parent.name.removeprefix('VisDrone2019-DET-'), images.parent)
    return found


def find_wu_test(base):
    """{'real': thư mục, 'synthetic': thư mục} có input/ và ground_truth/ của tập test Wu (bản Kaggle để hai phần ở
    hai thư mục gốc khác nhau)."""
    found = {}
    for split in ('real', 'synthetic'):
        for folder in glob_depth(base, split):
            if (folder / 'input').is_dir() and (folder / 'ground_truth').is_dir():
                found[split] = folder
                break
    return found


def label_file(root, stem):
    """Nhãn của một ảnh: annotations/<stem>.txt (định dạng VisDrone gốc, vd kushagrapandya/visdrone-dataset) hoặc
    labels/<stem>.txt (YOLO, vd banuprasadb/visdrone-dataset)."""
    for folder in ('annotations', 'labels'):
        if (root / folder).is_dir():
            return root / folder / f'{stem}.txt'
    return root / 'annotations' / f'{stem}.txt'


def count_vehicles(annotation):
    """Số hộp car, van, truck, bus. VisDrone gốc: 'x,y,w,h,score,category,...' với category 4, 5, 6, 9. YOLO (cách
    đổi của Ultralytics, category − 1): 'class cx cy w h' với class 3, 4, 5, 8."""
    vehicles = 0
    for line in Path(annotation).read_text(errors='replace').splitlines():
        line = line.strip()
        if ',' in line:
            values = line.rstrip(',').split(',')
            vehicles += len(values) >= 6 and values[5].strip() in ('4', '5', '6', '9')
        elif line:
            vehicles += line.split()[0] in ('3', '4', '5', '8')
    return vehicles


def day_scene_check(path, annotation, min_vehicles=5):
    """Cùng ngưỡng với visdrone-day-builder. Trả về lý do: 'qua', 'tối', 'nền tối', 'nhiều pixel tối', 'cháy sáng',
    'không có nhãn', 'ít xe' hoặc 'lỗi: ...' (không bao giờ ném lỗi, để Cell 4 đếm được vì sao ảnh bị loại)."""
    try:
        with Image.open(path) as image:
            width, height = image.size
            image.draft('RGB', (width // 4, height // 4))
            pixels = np.asarray(image.convert('RGB'), dtype=np.float32)
        luma = 0.299 * pixels[..., 0] + 0.587 * pixels[..., 1] + 0.114 * pixels[..., 2]
        if luma.mean() < 90:
            return 'tối'
        if np.percentile(luma, 10) < 25:
            return 'nền tối'
        if (luma < 40).mean() > 0.10:
            return 'nhiều pixel tối'
        if (luma > 250).mean() > 0.02:
            return 'cháy sáng'
        if not Path(annotation).is_file():
            return 'không có nhãn'
        vehicles = count_vehicles(annotation)
        return 'qua' if vehicles >= min_vehicles else 'ít xe'
    except Exception as exc:
        return f'lỗi: {type(exc).__name__}: {str(exc)[:120]}'


def flare_group(part, name):
    """Nhóm để chia valid: captured theo khối 10 khung liền nhau, simulated theo hình lá khẩu (như lens_flare.py)."""
    if part == 'captured':
        match = re.search(r'(\d+)', name)
        return f'c{int(match.group(1)) // 10 if match else 0}', (int(match.group(1)) // 10 if match else 0)
    match = re.search(r'aperture(\d+)', name)
    return f's{int(match.group(1)) if match else 0}', (int(match.group(1)) if match else 0)


def srgb_to_linear(x, gamma):
    return np.power(np.clip(x, 0.0, 1.0), gamma)


def linear_to_srgb(x, gamma):
    return np.power(np.clip(x, 0.0, 1.0), 1.0 / gamma)


def remove_background(im):
    """Bỏ nền DC, giữ mức trắng (utils.remove_background của Wu)."""
    low = im.min(axis=(0, 1), keepdims=True)
    high = im.max(axis=(0, 1), keepdims=True)
    return (im - low) * high / (high - low + 1e-7)


def normalize_white_balance(im):
    """Kênh nào cũng có trung bình bằng trung bình lớn nhất (utils.normalize_white_balance của Wu)."""
    mean = im.mean(axis=(0, 1), keepdims=True)
    return mean.max() * im / (mean + 1e-7)


def content_mask(pixels, feather):
    """1 trong phần ảnh thật của lớp lóe, mờ dần về 0 trong ``feather`` px tới mép của nó và mép khung.

    wu-lens-flare-640 đệm ảnh 640×477 thành 640×640 bằng màu nhỏ nhất; nền thật của ảnh chụp sáng hơn màu đó (sRGB
    ~13–25 so với ~6–10), nên sau khi nhân hệ số sáng tới 10, mép dải ảnh hiện thành một vạch thẳng. Ảnh gốc của Wu
    (1008×752) không gặp chuyện này vì ô cắt 512 luôn nằm trong ảnh."""
    pad = np.all(pixels == pixels.min(axis=(0, 1)), axis=-1)
    rows, cols = ~pad.all(axis=1), ~pad.all(axis=0)
    inside = np.zeros(pad.shape, np.uint8)
    inside[np.ix_(rows, cols)] = 1
    if feather <= 0:
        return inside.astype(np.float32)
    framed = np.pad(inside, 1)  # mép khung cũng là mép
    distance = cv2.distanceTransform(framed, cv2.DIST_L2, 5)[1:-1, 1:-1]
    ramp = np.clip(distance / feather, 0.0, 1.0)
    return (ramp * ramp * (3.0 - 2.0 * ramp)).astype(np.float32)  # smoothstep


def reveal_ghosts(flare, mask, boost, open_px, cap):
    """Làm sáng bóng ma và vòng quầng của lớp lóe tuyến tính (xem GHOST_BOOST ở Cell 1); tia sao và quầng chói tròn
    quanh lõi giữ nguyên. Nền (trung vị mức sáng trong phần ảnh thật) được trừ trước để không khuếch đại nền nhiễu."""
    level = flare.max(axis=-1)
    inside = mask > 0.5
    floor = float(np.median(level[inside])) if inside.any() else 0.0
    residual = np.clip(flare - floor, 0.0, None)
    level = residual.max(axis=-1)
    height, width = level.shape
    smooth = cv2.GaussianBlur(level, (0, 0), 4)
    if smooth.max() >= 0.5:  # có lõi sáng trong khung: quầng quanh lõi
        y, x = np.unravel_index(int(np.argmax(smooth)), smooth.shape)
    else:  # nguồn ngoài khung: quầng quanh tâm
        y, x = height / 2.0, width / 2.0
    yy, xx = np.mgrid[:height, :width]
    radius = np.hypot(yy - y, xx - x).astype(np.int32).ravel()
    order = np.argsort(radius, kind='stable')
    bounds = np.searchsorted(radius[order], np.arange(radius.max() + 2))
    values = level.ravel()[order]
    profile = np.array([np.median(values[a:b]) if b > a else 0.0 for a, b in zip(bounds[:-1], bounds[1:])],
                       dtype=np.float32)
    glare = profile[radius].reshape(height, width)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_px, open_px))
    blobs = cv2.morphologyEx(level, cv2.MORPH_OPEN, kernel)
    ghost = cv2.GaussianBlur(np.clip(blobs - glare, 0.0, None), (0, 0), 2)
    added = np.maximum(np.minimum(boost * ghost, cap), ghost) - ghost  # tối đa tới cap; bóng ma sáng sẵn: thêm 0
    colour = residual / np.maximum(level, 1e-6)[..., None]  # màu của lớp lóe ở chỗ đó, kênh lớn nhất = 1
    return residual + added[..., None] * colour


def affine(im, rng, scale_range=None, translate=None):
    """Xoay, nghiêng, co giãn quanh tâm rồi dịch (utils.apply_affine_transform của Wu), song tuyến, nền 0.
    Mặc định là tham số của Wu (dịch N(0, SHIFT_STD)); ``translate`` (tỉ lệ cạnh) cho dịch đều như FlareX."""
    height, width = im.shape[:2]
    angle = math.radians(rng.uniform(-ROTATION, ROTATION))
    shift = (rng.normal(0.0, SHIFT_STD, 2) if translate is None
             else rng.uniform(-translate, translate, 2) * np.array([width, height]))
    shear = np.tan(np.radians(rng.uniform(-SHEAR, SHEAR, 2)))
    scale = rng.uniform(*(scale_range or SCALE), 2)
    rotate = np.array([[math.cos(angle), -math.sin(angle)], [math.sin(angle), math.cos(angle)]])
    matrix = rotate @ np.array([[1.0, shear[0]], [shear[1], 1.0]]) @ np.diag(scale)
    center = np.array([width / 2.0, height / 2.0])
    offset = center - matrix @ center + shift
    warp = np.hstack([matrix, offset[:, None]]).astype(np.float32)
    channels = [cv2.warpAffine(np.ascontiguousarray(im[..., i:i + 4]), warp, (width, height), flags=cv2.INTER_LINEAR,
                               borderMode=cv2.BORDER_CONSTANT, borderValue=0) for i in range(0, im.shape[2], 4)]
    out = np.dstack([c if c.ndim == 3 else c[..., None] for c in channels])  # OpenCV: tối đa 4 kênh mỗi lần
    return out, {'rotation': math.degrees(angle), 'shift': shift.round(2).tolist(),
                 'shear': np.degrees(np.arctan(shear)).round(2).tolist(), 'scale': scale.round(3).tolist()}


def center_crop(im, size):
    height, width = im.shape[:2]
    if height < size or width < size:  # lớp lóe nhỏ hơn đầu ra: đệm 0 (nền đã bỏ DC)
        padded = np.zeros((max(height, size), max(width, size), im.shape[2]), im.dtype)
        top, left = (padded.shape[0] - height) // 2, (padded.shape[1] - width) // 2
        padded[top:top + height, left:left + width] = im
        im, height, width = padded, padded.shape[0], padded.shape[1]
    top, left = (height - size) // 2, (width - size) // 2
    return im[top:top + size, left:left + size]


def random_flips(rng, *images):
    flip_x, flip_y = rng.random() < 0.5, rng.random() < 0.5
    out = []
    for im in images:
        im = im[:, ::-1] if flip_x else im
        out.append(np.ascontiguousarray(im[::-1] if flip_y else im))
    return out


def load_scene(path, rng, upscale=False):
    """Ảnh sạch sRGB [0,1], cạnh ngắn SCENE_SHORT, cắt ngẫu nhiên OUT_SIZE, lật ngẫu nhiên. ``upscale``: ảnh nhỏ hơn
    cũng đưa cạnh ngắn về SCENE_SHORT (ảnh Flickr; VisDrone thì không phóng to, ảnh nhỏ đã bị bỏ ở Cell 4)."""
    with Image.open(path) as image:
        image = image.convert('RGB')
        scale = SCENE_SHORT / min(image.size)
        if scale < 1 or (upscale and scale > 1):
            image = image.resize((max(OUT_SIZE, round(image.width * scale)), max(OUT_SIZE, round(image.height * scale))),
                                 Image.Resampling.LANCZOS)
        pixels = np.asarray(image, dtype=np.float32) / 255.0
    height, width = pixels.shape[:2]
    if height < OUT_SIZE or width < OUT_SIZE:
        raise ValueError(f'{path}: {width}×{height} nhỏ hơn OUT_SIZE {OUT_SIZE}')
    top, left = int(rng.integers(0, height - OUT_SIZE + 1)), int(rng.integers(0, width - OUT_SIZE + 1))
    crop = pixels[top:top + OUT_SIZE, left:left + OUT_SIZE]
    return random_flips(rng, crop)[0], (left, top)


def load_linear(path, gamma, side=None):
    """Ảnh RGB đọc ra ánh sáng tuyến tính (x^gamma), thu cạnh dài về ``side`` bằng trung bình diện tích (INTER_AREA)."""
    with Image.open(path) as image:
        pixels = np.asarray(image.convert('RGB'), dtype=np.float32) / 255.0
    linear = srgb_to_linear(pixels, gamma)
    if side and max(linear.shape[:2]) > side:
        scale = side / max(linear.shape[:2])
        size = (max(1, round(linear.shape[1] * scale)), max(1, round(linear.shape[0] * scale)))
        linear = cv2.resize(linear, size, interpolation=cv2.INTER_AREA)
    return linear


def compose(scene_path, flare, seed, upscale=False):
    """Một cặp: (ảnh có lóe, ảnh đích, lớp lóe, tham số), uint8 [S,S,3]. ``flare``: hàng của FLARES (Cell 4);
    ``upscale``: như load_scene.
    Lớp Wu theo synthesis.add_flare của Wu et al.; lớp FlareX theo flare7kpp_dataset.py của FlareX."""
    rng = np.random.default_rng(seed)
    gamma = float(rng.uniform(*GAMMA))
    flare_path, part = flare['path'], flare['part']
    if part == 'flarex':
        # Lớp lóe và ảnh nguồn sáng của nó đi cùng phép biến hình; nguồn sáng không bỏ nền, không nhân độ sáng.
        layer = remove_background(load_linear(flare_path, gamma, FLAREX_SIDE))
        source = load_linear(flare['light'], gamma, FLAREX_SIDE)
        stacked, geometry = affine(np.dstack([layer, source]).astype(np.float32), rng, FLAREX_SCALE, FLAREX_TRANSLATE)
        stacked = random_flips(rng, center_crop(np.clip(stacked, 0.0, 1.0), OUT_SIZE))[0]
        flare_lin, light_lin = stacked[..., :3], stacked[..., 3:]
        brightness = float(rng.uniform(*FLAREX_BRIGHTNESS))
        if FLAREX_MAX_MEAN:  # lớp phủ kín khung: giảm cho lượng lóe trung bình không quá trần
            brightness = min(brightness, FLAREX_MAX_MEAN / max(float((flare_lin - light_lin).clip(0).mean()), 1e-6))
        gains = np.full(3, brightness, np.float32)
        light_lin = light_lin * min(brightness, 1.0)  # lóe bị giảm thì nguồn sáng giảm theo (FlareX: độ sáng ≥ 0,8)
        ghost = 1
    else:
        with Image.open(flare_path) as image:
            flare_u8 = np.asarray(image.convert('RGB'))
        flare_lin = remove_background(srgb_to_linear(flare_u8.astype(np.float32) / 255.0, gamma))
        mask = content_mask(flare_u8, FEATHER)
        flare_lin = flare_lin * mask[..., None]
        # Lõi nguồn sáng đo TRƯỚC khi làm sáng bóng ma và nhân hệ số sáng (như wu_light của repo đo trên ảnh 8 bit
        # gốc): đo sau thì quầng, tia và bóng ma đã sáng lên đều vượt 225 và bị coi là "nguồn sáng" ở ảnh đích.
        core = linear_to_srgb(flare_lin, gamma).max(axis=-1) * 255.0
        light = np.clip((core - LIGHT_LOW) / (LIGHT_HIGH - LIGHT_LOW), 0.0, 1.0)
        ghost = GHOST_BOOST if part in GHOST_PARTS else 1
        if ghost != 1:
            flare_lin = reveal_ghosts(flare_lin, mask, ghost, GHOST_OPEN, GHOST_CAP) * mask[..., None]
        stacked, geometry = affine(np.dstack([flare_lin, light]).astype(np.float32), rng)  # lõi đi cùng biến hình
        stacked = random_flips(rng, center_crop(np.clip(stacked, 0.0, 1.0), OUT_SIZE))[0]
        flare_lin, light_weight = stacked[..., :3], stacked[..., 3]
        flare_lin = normalize_white_balance(flare_lin)
        if FLARE_GAIN_MODE == 'per_channel':
            gains = rng.uniform(0.0, FLARE_MAX_GAIN, 3)
        else:
            gains = rng.uniform(FLARE_MIN_GAIN, FLARE_MAX_GAIN) * rng.uniform(0.8, 1.2, 3)
    flare_lin = flare_lin * gains.astype(np.float32)
    sigma = float(rng.uniform(*BLUR_SIGMA))
    flare_lin = cv2.GaussianBlur(flare_lin, (21, 21), sigma, borderType=cv2.BORDER_CONSTANT)
    offset = 0.0 if part == 'flarex' else float(rng.uniform(-DC_OFFSET, DC_OFFSET))  # FlareX không cộng offset DC
    flare_lin = np.clip(flare_lin + offset, 0.0, 1.0)

    scene, crop_at = load_scene(scene_path, rng, upscale)
    scene = srgb_to_linear(scene, gamma)
    noise_sigma = abs(float(rng.normal(0.0, NOISE)))
    scene = scene + rng.normal(0.0, noise_sigma, scene.shape).astype(np.float32)
    scene_gain = float(rng.uniform(*SCENE_GAIN))
    scene = np.clip(scene_gain * scene, 0.0, 1.0)

    flare_srgb = linear_to_srgb(flare_lin, gamma)
    combined = linear_to_srgb(scene + flare_lin, gamma)
    if not KEEP_LIGHT:
        target = linear_to_srgb(scene, gamma)
    elif part == 'flarex':  # nguồn sáng là ảnh gt/ của FlareX, không sáng hơn lớp lóe (ảnh đích ≤ ảnh vào)
        target = linear_to_srgb(scene + np.minimum(light_lin, flare_lin), gamma)
    else:  # lõi nguồn sáng: phần gần bão hoà của lớp lóe trước khi nhân hệ số (wu_light của repo)
        weight = cv2.GaussianBlur(light_weight, (21, 21), sigma, borderType=cv2.BORDER_CONSTANT)  # cùng độ mờ lớp lóe
        target = linear_to_srgb(scene + flare_lin * weight[..., None], gamma)
    to_u8 = lambda x: np.round(np.clip(x, 0.0, 1.0) * 255.0).astype(np.uint8)
    params = {'source': part, 'gamma': round(gamma, 4), 'ghost_boost': ghost, 'flare_gain': gains.round(3).tolist(),
              'scene_gain': round(scene_gain, 4), 'blur_sigma': round(sigma, 3), 'dc_offset': round(offset, 4),
              'noise_sigma': round(noise_sigma, 5), 'crop': crop_at, **geometry}
    return to_u8(combined), to_u8(target), to_u8(flare_srgb), params


def find_backgrounds(base):
    """Ảnh nền Flickr24K đã gắn làm Input: ('dir', thư mục Flickr24K/ có ≥ 1 000 ảnh) hoặc ('zip', file zip có tên chứa
    flickr24k), không có thì None."""
    for folder in glob_depth(base, 'Flickr24K'):
        if folder.is_dir() and sum(1 for p in folder.rglob('*') if p.suffix.lower() in IMAGE_SUFFIXES) >= 1000:
            return 'dir', folder
    for path in glob_depth(base, '*.zip'):
        if 'flickr24k' in path.name.lower().replace('_', '').replace('-', '') and zipfile.is_zipfile(path):
            return 'zip', path
    return None


def background_check(path):
    """(lý do, độ nét) của một ảnh nền: 'qua', 'nhỏ', 'tối', 'đen trắng', 'cháy sáng' hoặc 'lỗi: ...'. Độ nét = phương
    sai Laplace của kênh sáng ở cạnh ngắn 384 (cỡ repo đọc cặp khi train). Không bao giờ ném lỗi."""
    try:
        with Image.open(path) as image:
            if min(image.size) < BACKGROUND_MIN_SHORT:
                return 'nhỏ', 0.0
            image = image.convert('RGB')
            scale = 384 / min(image.size)
            image = image.resize((max(1, round(image.width * scale)), max(1, round(image.height * scale))),
                                 Image.Resampling.BOX if scale < 1 else Image.Resampling.LANCZOS)
            pixels = np.asarray(image, dtype=np.float32)
        luma = 0.299 * pixels[..., 0] + 0.587 * pixels[..., 1] + 0.114 * pixels[..., 2]
        chroma = np.abs(pixels[..., 0] - pixels[..., 1]).mean() + np.abs(pixels[..., 1] - pixels[..., 2]).mean()
        sharpness = float(cv2.Laplacian(luma, cv2.CV_32F).var())
        if luma.mean() < BACKGROUND_MIN_LUMA:
            return 'tối', sharpness
        if chroma < BACKGROUND_MIN_CHROMA:
            return 'đen trắng', sharpness
        if (luma > 250).mean() > BACKGROUND_MAX_SATURATED:
            return 'cháy sáng', sharpness
        return 'qua', sharpness
    except Exception as exc:
        return f'lỗi: {type(exc).__name__}: {str(exc)[:120]}', 0.0


def find_flarex(base):
    """Thư mục Flare2D/ của FlareX có input/ (lớp lóe) và gt/ (nguồn sáng)."""
    for folder in glob_depth(base, 'Flare2D', depth=7):
        if (folder / 'input').is_dir() and (folder / 'gt').is_dir():
            return folder
    return None


def flare_strength(lq, gt):
    """Độ lóe của một cặp uint8: (lóe trung bình tuyến tính, tỉ lệ pixel tăng > 0,1, tỉ lệ pixel bão hoà của ảnh vào)."""
    lq_lin = srgb_to_linear(lq.astype(np.float32) / 255.0, 2.2)
    gt_lin = srgb_to_linear(gt.astype(np.float32) / 255.0, 2.2)
    added = np.clip(lq_lin - gt_lin, 0.0, None).mean(axis=-1)
    return float(added.mean()), float((added > 0.1).mean()), float((lq.mean(axis=-1) >= 252).mean())


def pair_sheet(pairs, path, title, thumb=240, columns=4):
    """Lưới cặp (vào → đích) như 01_wu_test_that.png; ``pairs`` = [(nhãn, ảnh vào, ảnh đích)] uint8."""
    from PIL import ImageDraw, ImageFont
    try:
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 13)
    except OSError:
        font = ImageFont.load_default()
    rows = math.ceil(len(pairs) / columns)
    sheet = Image.new('RGB', (columns * 2 * thumb, 28 + rows * (thumb + 20)), (255, 255, 255))
    draw = ImageDraw.Draw(sheet)
    draw.text((4, 6), title, fill=(0, 0, 0), font=font)
    for k, (label, lq, gt) in enumerate(pairs):
        x, y = (k % columns) * 2 * thumb, 28 + (k // columns) * (thumb + 20)
        for j, image in enumerate((lq, gt)):
            im = Image.fromarray(image)
            im.thumbnail((thumb, thumb), Image.Resampling.BOX)
            sheet.paste(im, (x + j * thumb, y + 20))
        draw.text((x + 3, y + 3), label, fill=(0, 0, 0), font=font)
    sheet.save(path)
    return sheet


def compose_job(job):
    """Tiến trình con: ghép một cặp và ghi PNG; trả về hàng index.csv hoặc lỗi."""
    index, scene, flare, seed, split, out_dir = job
    try:
        lq, gt, _, params = compose(scene['path'], flare, seed, scene['upscale'])
        if not lq.shape == gt.shape == (OUT_SIZE, OUT_SIZE, 3):
            raise ValueError(f'cặp {lq.shape} / {gt.shape}, không phải {OUT_SIZE}×{OUT_SIZE}')
        name = f'{index:05d}.png'
        Image.fromarray(lq).save(Path(out_dir) / 'input' / name, compress_level=6)
        Image.fromarray(gt).save(Path(out_dir) / 'gt' / name, compress_level=6)
        strength = flare_strength(lq, gt)
        return {'file': name, 'split': split, 'scene': scene['name'], 'scene_split': scene['split'],
                'flare': f'{flare["part"]}/{flare["name"]}', 'seed': seed, 'flare_mean': round(strength[0], 5),
                'flare_area': round(strength[1], 4), 'saturated': round(strength[2], 4),
                'params': json.dumps(params)}
    except Exception as exc:
        return {'file': None, 'error': f'{type(exc).__name__}: {exc}', 'scene': scene['name']}


LIST_FILES = r"""
import json, sys
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()
token = None
while True:
    response = api.dataset_list_files(sys.argv[1], token, int(sys.argv[2]))
    if response is None or getattr(response, "error_message", None):
        print(json.dumps({"error": getattr(response, "error_message", None) or "no response"}))
        break
    for item in response.files or []:
        print(json.dumps({"name": item.name, "bytes": getattr(item, "total_bytes", None)}))
    token = response.next_page_token
    if not token:
        break
"""


def remote_files(dataset, env, page_size=200):
    code, out, err = run([sys.executable, '-c', LIST_FILES, dataset, str(page_size)], env=env, timeout=1800)
    rows = [json.loads(line) for line in out.splitlines() if line.startswith('{')]
    errors = [row['error'] for row in rows if 'error' in row]
    if code != 0 or errors:
        raise RuntimeError(f'liệt kê file thất bại (mã {code}): {errors or err.strip()[-1500:]}')
    return rows


print('đã định nghĩa hàm')

## Cell 3 — Khoá Kaggle (không bắt buộc)

Chỉ cần khi muốn Cell 8 tự tạo dataset. Không có khoá thì vẫn xem trước được và đóng gói vào `/kaggle/working`.

In [ ]:
import importlib.metadata

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
except ImportError:  # ngoài Kaggle: đọc biến môi trường
    _secrets = None
secret_errors = {}


def secret(name):
    try:
        return _secrets.get_secret(name) if _secrets else os.environ.get(name)
    except Exception as error:  # thông báo không chứa giá trị secret
        secret_errors[name] = f'{type(error).__name__}: {str(error)[:300]}'
        return None


_token = secret('KAGGLE_API_TOKEN')
_key = None if _token else secret('KAGGLE_KEY')
OWNER = KAGGLE_OWNER or (None if _token else secret('KAGGLE_USERNAME'))
UPLOAD_BY_API = bool(_token or (_key and OWNER))
ENV, DATASET, EXISTS = None, None, False
if UPLOAD_BY_API:
    code_, out, err = run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', 'kaggle>=2.2'])
    assert code_ == 0, f'pip install lỗi:\n{err[-2000:]}'
    # Khoá chỉ nằm trong ENV, truyền cho tiến trình con kaggle; bỏ khoá có sẵn để không đăng nhập nhầm tài khoản.
    ENV = {name: value for name, value in os.environ.items()
           if name not in ('KAGGLE_API_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY')}
    ENV.update({'KAGGLE_API_TOKEN': _token} if _token else {'KAGGLE_USERNAME': OWNER, 'KAGGLE_KEY': _key})
del _key, _token
if UPLOAD_BY_API and not OWNER:
    code_, out, err = run([sys.executable, '-c', 'from kaggle.api.kaggle_api_extended import KaggleApi; api = KaggleApi(); '
                           'api.authenticate(); print(api.get_config_value("username") or "")'], env=ENV, timeout=120)
    OWNER = (out.strip().splitlines() or [''])[-1].strip()
    if code_ != 0 or not OWNER:
        raise SystemExit(f'KAGGLE_API_TOKEN không đăng nhập được (token sai hoặc hết hạn?): {err.strip()[-500:]}')
if UPLOAD_BY_API:
    DATASET = f'{OWNER}/{SLUG}'
    code_, text = kaggle('datasets', 'status', DATASET, env=ENV)
    EXISTS = code_ == 0 and not any(word in text.lower() for word in ('404', 'not found'))
    print(f'kaggle {importlib.metadata.version("kaggle")} | tài khoản {OWNER} | {DATASET}: '
          f'{"đã có, Cell 8 sẽ thêm version mới" if EXISTS else "chưa có, Cell 8 sẽ tạo mới"}')
else:
    for name, error in secret_errors.items():
        print(f'{name}: {error}')
    print('Không có khoá: vẫn xem trước được; Cell 8 sẽ đóng gói vào /kaggle/working để tạo dataset bằng tay.')

## Cell 4 — Tìm lớp lóe, ảnh VisDrone, tập test Wu

Lớp lóe được chia train/valid theo nhóm (khối 10 khung của `captured/`, hình lá khẩu của `simulated/`): nhóm thứ 0,
`VAL_EVERY`, 2·`VAL_EVERY`… vào valid. Ảnh VisDrone (không bắt buộc): ảnh từ test-dev vào valid (không chung chuỗi
với train/val).

In [ ]:
WU_ROOT = find_wu_flares(INPUT)
assert WU_ROOT, 'Không thấy lớp lóe Wu (captured/ + simulated/): Add Input → buidinhkhoi/wu-lens-flare-640'
FLARES = {'train': [], 'val': []}
for part in ('captured', 'simulated'):
    for path in sorted((WU_ROOT / part).glob('*.png')):
        group, number = flare_group(part, path.name)
        FLARES['val' if number % VAL_EVERY == 0 else 'train'].append(
            {'part': part, 'name': path.name, 'path': str(path), 'group': group})
assert set(FLARE_SHARES) <= {'captured', 'simulated', 'flarex'} and abs(sum(FLARE_SHARES.values()) - 1) < 1e-9, \
    f'FLARE_SHARES: chỉ captured / simulated / flarex, tổng bằng 1 ({FLARE_SHARES})'
FLAREX_ROOT = find_flarex(INPUT) if FLARE_SHARES.get('flarex', 0) > 0 else None
if FLAREX_ROOT:
    # Ghép lóe với nguồn sáng theo tên (FlareX ghép theo thứ tự tên đã sắp xếp; tên ở hai thư mục trùng nhau).
    # Valid: các "loại" lóe thứ 0, VAL_EVERY, ... (mỗi loại 100 ảnh liền số: 0001–0100 là loại 0, ...).
    lights = {path.name for path in (FLAREX_ROOT / 'gt').iterdir() if path.suffix.lower() in IMAGE_SUFFIXES}
    for path in sorted(p for p in (FLAREX_ROOT / 'input').iterdir() if p.suffix.lower() in IMAGE_SUFFIXES):
        if path.name not in lights:
            continue
        number = int(re.sub(r'\D', '', path.stem) or 0)
        kind = max(number - 1, 0) // 100
        FLARES['val' if kind % VAL_EVERY == 0 else 'train'].append(
            {'part': 'flarex', 'name': path.name, 'path': str(path), 'light': str(FLAREX_ROOT / 'gt' / path.name),
             'group': f'x{kind}'})
print(f'lớp lóe: Wu {WU_ROOT}' + (f' + FlareX {FLAREX_ROOT}' if FLAREX_ROOT else
                                  ' (FlareX: không gắn; tuỳ chọn Add Input → lishenqu/flarex)'))
for split in ('train', 'val'):
    print(f'  {split}: {dict(Counter(f["part"] for f in FLARES[split]))}')

SCENES = []
DAY_ROOT = find_visdrone_day(INPUT)
RAW = find_visdrone_raw(INPUT)
USED_SCENES, USED_ROOTS = find_used_scenes(INPUT) if VISDRONE_AVOID_USED else (set(), [])
if DAY_ROOT and not RAW:
    with open(DAY_ROOT / 'index.csv') as handle:
        for row in csv.DictReader(handle):
            SCENES.append({'name': row['file'], 'path': str(DAY_ROOT / 'images' / row['file']),
                           'split': row['source_split'], 'sequence': row['sequence']})
    print(f'ảnh sạch: {DAY_ROOT} ({len(SCENES)} ảnh, visdrone-day-builder)')
elif RAW:
    candidates = [(split, path, label_file(root, path.stem)) for split, root in sorted(RAW.items())
                  if split in ('train', 'val', 'test-dev') for path in sorted((root / 'images').glob('*.jpg'))]
    for split, root in sorted(RAW.items()):
        labels = label_file(root, 'x').parent
        print(f'  {split:<15} {root} | images/ {len(list((root / "images").glob("*.jpg")))} | {labels.name}/ '
              f'{len(list(labels.glob("*.txt"))) if labels.is_dir() else "KHÔNG CÓ"}')
    # Thử 40 ảnh trước (vài giây): không ảnh nào qua thì dừng ngay, in lý do, khỏi chờ quét hết ~10 phút.
    sample = [candidates[i] for i in np.random.default_rng(0).choice(len(candidates), min(40, len(candidates)), replace=False)]
    sample_reasons = Counter(day_scene_check(path, annotation) for _, path, annotation in sample)
    print(f'  thử {len(sample)} ảnh: {dict(sample_reasons)}')
    assert sample_reasons['qua'] > 0, (f'{len(sample)} ảnh thử không ảnh nào qua: {dict(sample_reasons)}. Ví dụ ảnh '
                                      f'{sample[0][1]} (có: {sample[0][1].is_file()}), nhãn {sample[0][2]} '
                                      f'(có: {sample[0][2].is_file()}). Gửi dòng này cho mình.')
    cache = Path('/tmp/visdrone_day_scenes.json')  # quét một lần mỗi phiên; chạy lại Cell 4 thì đọc lại
    key = [len(candidates), str(candidates[0][1]), str(candidates[-1][1]), str(candidates[0][2])]
    cached = json.loads(cache.read_text()) if cache.is_file() else None
    if cached and cached.get('key') == key:
        SCENES, reasons = cached['scenes'], Counter(cached['reasons'])
        print(f'  đọc lại kết quả lọc của phiên này từ {cache}')
    else:
        started = time.perf_counter()
        with ProcessPoolExecutor(max_workers=os.cpu_count() or 4) as pool:
            verdicts = list(pool.map(day_scene_check, [c[1] for c in candidates], [c[2] for c in candidates],
                                     chunksize=64))
        reasons = Counter(verdict.split(':')[0] for verdict in verdicts)
        SCENES = [{'name': path.name, 'path': str(path), 'split': split, 'sequence': path.name.split('_')[0]}
                  for (split, path, _), verdict in zip(candidates, verdicts) if verdict == 'qua']
        errors = [verdict for verdict in verdicts if verdict.startswith('lỗi')]
        for error in errors[:3]:
            print('  ', error)
        cache.write_text(json.dumps({'key': key, 'scenes': SCENES, 'reasons': reasons}))
        print(f'  quét {len(candidates)} ảnh trong {time.perf_counter() - started:.0f} s')
    print(f'ảnh sạch: VisDrone gốc, lọc ban ngày có xe: {len(SCENES)}/{len(candidates)} ảnh | lý do: {dict(reasons)}')
    assert SCENES, 'Gắn VisDrone mà không ảnh nào qua lọc; xem dòng lý do ở trên'
else:
    print('ảnh VisDrone: không gắn (tuỳ chọn: visdrone-day-2000 hoặc kushagrapandya/visdrone-dataset); '
          'mọi cặp dùng ảnh nền Flickr24K (Cell 4b)')


def short_side(path):
    with Image.open(path) as image:  # chỉ đọc phần đầu file
        return min(image.size)


small = [scene for scene in SCENES if short_side(scene['path']) < OUT_SIZE]
if small:
    names = {scene['path'] for scene in small}
    SCENES = [scene for scene in SCENES if scene['path'] not in names]
    print(f'bỏ {len(small)} ảnh có cạnh ngắn < OUT_SIZE {OUT_SIZE} (không phóng to ảnh sạch), còn {len(SCENES)}')
for scene in SCENES:
    scene.update(source='visdrone', upscale=False)
if USED_ROOTS:
    print(f'ảnh VisDrone bộ cũ đã dùng: {len(USED_SCENES)} (đọc từ {USED_ROOTS}); trong số ảnh có ở đây: '
          f'{sum(s["name"] in USED_SCENES for s in SCENES)} đã dùng, {sum(s["name"] not in USED_SCENES for s in SCENES)} chưa')
elif VISDRONE_AVOID_USED and SCENES:
    raise SystemExit('Có ảnh VisDrone mà không gắn bộ cũ (buidinhkhoi/visdrone-lens-flare-640 hoặc visdrone-day-2000): '
                     'không biết ảnh nào đã dùng, cặp VisDrone sẽ trùng bộ cũ (lần dựng đầu: 633/1 000). Add Input '
                     '→ visdrone-lens-flare-640 rồi chạy lại Cell 4, hoặc đặt VISDRONE_AVOID_USED = False ở Cell 1.')
SCENES_BY_SPLIT = {'train': [s for s in SCENES if s['split'] != 'test-dev'],
                   'val': [s for s in SCENES if s['split'] == 'test-dev']}
print(f'ảnh VisDrone train {len(SCENES_BY_SPLIT["train"])}, valid {len(SCENES_BY_SPLIT["val"])}')

WU_TEST = find_wu_test(INPUT)
WU_TEST_PAIRS = {}
if WU_TEST:
    for split, folder in WU_TEST.items():
        names = sorted(path.name for path in (folder / 'input').iterdir() if path.suffix.lower() in IMAGE_SUFFIXES)
        WU_TEST_PAIRS[split] = [(name, folder / 'input' / name, folder / 'ground_truth' / name)
                                for name in names if (folder / 'ground_truth' / name).is_file()]
    print(f'tập test Wu: {sorted(map(str, WU_TEST.values()))} | ' + ', '.join(f'{k} {len(v)} cặp' for k, v in WU_TEST_PAIRS.items()))
else:
    print('tập test Wu: không gắn (tuỳ chọn: thomasdanielardhali/how-to-train-nn-for-flare-removal-dataset)')


BY_PART = {split: {part: [f for f in FLARES[split] if f['part'] == part]
                    for part in ('captured', 'simulated', 'flarex')} for split in FLARES}


# Tỉ lệ thật: các phần có lớp lóe ở cả train lẫn valid (FlareX chỉ khi đã gắn), chia lại cho tổng bằng 1.
present = {part: share for part, share in FLARE_SHARES.items()
           if share > 0 and BY_PART['train'][part] and BY_PART['val'][part]}
assert present, f'Không có lớp lóe nào cho FLARE_SHARES {FLARE_SHARES}'
FLARE_USED = {part: share / sum(present.values()) for part, share in present.items()}
missing = sorted(part for part, share in FLARE_SHARES.items() if share > 0 and part not in FLARE_USED)
print('tỉ lệ lớp lóe: ' + ', '.join(f'{part} {share:.0%}' for part, share in FLARE_USED.items())
      + (f' (thiếu {missing}: chia lại phần của nó; FlareX: Add Input → lishenqu/flarex)' if missing else ''))


def flare_parts(rng, count):
    """``count`` nguồn lóe đúng theo FLARE_USED (làm tròn kiểu phần dư lớn nhất), xáo ngẫu nhiên."""
    parts = sorted(FLARE_USED)
    exact = np.array([FLARE_USED[part] * count for part in parts])
    counts = np.floor(exact).astype(int)
    for index in np.argsort(-(exact - counts), kind='stable')[:count - counts.sum()]:
        counts[index] += 1
    order = [part for part, n in zip(parts, counts) for _ in range(n)]
    return [order[i] for i in rng.permutation(len(order))]


def pick_flare(rng, split, part=None):
    """Một lớp lóe của ``split``: nguồn ``part`` (Cell 6 chia đúng tỉ lệ) hoặc rút theo FLARE_USED (xem trước)."""
    if part is None:
        parts = sorted(FLARE_USED)
        part = parts[int(rng.choice(len(parts), p=[FLARE_USED[p] for p in parts]))]
    choices = BY_PART[split][part]
    return choices[int(rng.integers(len(choices)))]

## Cell 4b — Ảnh nền Flickr24K (tải, rút, lọc)

Gắn sẵn trên Kaggle (thư mục `Flickr24K/` hay file zip) thì đọc thẳng; không thì `gdown` tải `Flickr24K.zip` từ link
Google Drive chính thức của Flare7K vào `/tmp` (thử lại khi Drive hết lượt tải; vẫn lỗi thì xem `BACKGROUND_DRIVE_ID`
ở Cell 1). Từ zip chỉ giải nén `BACKGROUND_CANDIDATES` ảnh rút ngẫu nhiên (`SEED`). Lọc: cạnh ngắn ≥
`BACKGROUND_MIN_SHORT`, không tối, không đen trắng, không cháy sáng, rồi bỏ `BACKGROUND_DROP_BLURRY` ảnh mờ nhất. Ảnh
thứ 0, `VAL_EVERY`, 2·`VAL_EVERY`… (theo tên) vào valid. Zip giữ lại trong `/tmp`: chạy lại cell này không tải lại.

In [ ]:
BACKGROUNDS = []
if BACKGROUND_FRACTION > 0:
    found = find_backgrounds(INPUT)
    if found and found[0] == 'dir':
        files = sorted(p for p in found[1].rglob('*') if p.suffix.lower() in IMAGE_SUFFIXES)
        print(f'ảnh nền: {found[1]} ({len(files)} ảnh, gắn làm Input)')
        picked = [files[i] for i in sorted(np.random.default_rng(SEED).permutation(len(files))[:BACKGROUND_CANDIDATES])]
    else:
        archive_path = found[1] if found else BACKGROUND_ZIP
        if not found and not (BACKGROUND_ZIP.is_file() and zipfile.is_zipfile(BACKGROUND_ZIP)):
            try:
                import gdown
            except ImportError:
                code_, out, err = run([sys.executable, '-m', 'pip', 'install', '--quiet', 'gdown'])
                assert code_ == 0, f'pip install gdown lỗi:\n{err[-2000:]}'
                import gdown
            from gdown.exceptions import FileURLRetrievalError
            for attempt in range(1 + DOWNLOAD_RETRIES):
                try:
                    if gdown.download(id=BACKGROUND_DRIVE_ID, output=str(BACKGROUND_ZIP), quiet=False, resume=True) \
                            and zipfile.is_zipfile(BACKGROUND_ZIP):
                        break
                    error = 'gdown không trả về file zip'
                except FileURLRetrievalError as exc:
                    error = str(exc)
                except Exception as exc:  # lỗi mạng giữa chừng: resume=True tải tiếp phần còn thiếu
                    error = f'{type(exc).__name__}: {exc}'
                quota = any(text in error.lower() for text in ('too many users', 'quota', 'many accesses'))
                print(f'Lần {attempt + 1} thất bại{" (Google Drive hết lượt tải)" if quota else ""}: {error[:600]}')
                if attempt < DOWNLOAD_RETRIES:
                    print(f'Chờ {RETRY_WAIT // 60} phút rồi thử lại ({attempt + 1}/{DOWNLOAD_RETRIES})', flush=True)
                    time.sleep(RETRY_WAIT)
            else:
                raise SystemExit(f'Không tải được Flickr24K ({BACKGROUND_DRIVE_ID}). Tạo bản sao file vào Google Drive '
                                 'của bạn, chia sẻ "bất kỳ ai có đường liên kết", dán ID bản sao vào BACKGROUND_DRIVE_ID '
                                 'ở Cell 1; hoặc đặt BACKGROUND_FRACTION = 0 để chỉ dùng VisDrone.')
        print(f'ảnh nền: {archive_path} ({gb(Path(archive_path).stat().st_size)})')
        if BACKGROUND_DIR.exists():
            shutil.rmtree(BACKGROUND_DIR)
        BACKGROUND_DIR.mkdir(parents=True)
        with zipfile.ZipFile(archive_path) as archive:
            members = sorted((m for m in archive.infolist() if not m.is_dir() and '__MACOSX' not in m.filename
                              and Path(m.filename).suffix.lower() in IMAGE_SUFFIXES), key=lambda m: m.filename)
            chosen = [members[i] for i in sorted(np.random.default_rng(SEED).permutation(len(members))[:BACKGROUND_CANDIDATES])]
            picked = []
            for member in chosen:
                # Tên phẳng, không thư mục con: không ghi ra ngoài BACKGROUND_DIR dù tên trong zip là gì.
                target = BACKGROUND_DIR / re.sub(r'[^A-Za-z0-9._-]', '_', member.filename.replace('/', '__'))
                with archive.open(member) as source, open(target, 'wb') as sink:
                    shutil.copyfileobj(source, sink)
                picked.append(target)
        print(f'  zip có {len(members)} ảnh, giải nén {len(picked)} ảnh ứng viên vào {BACKGROUND_DIR}')
    started = time.perf_counter()
    with ProcessPoolExecutor(max_workers=os.cpu_count() or 4) as pool:
        verdicts = list(pool.map(background_check, map(str, picked), chunksize=32))
    reasons = Counter(reason.split(':')[0] for reason, _ in verdicts)
    for reason, _ in [v for v in verdicts if v[0].startswith('lỗi')][:3]:
        print('  ', reason)
    passed = [(path, sharpness) for path, (reason, sharpness) in zip(picked, verdicts) if reason == 'qua']
    cut = float(np.quantile([s for _, s in passed], BACKGROUND_DROP_BLURRY)) if passed and BACKGROUND_DROP_BLURRY else 0.0
    kept = sorted((path for path, sharpness in passed if sharpness >= cut), key=lambda p: p.name)
    reasons['mờ'] = len(passed) - len(kept)
    reasons['qua'] = len(kept)
    for k, path in enumerate(kept):
        BACKGROUNDS.append({'name': path.name, 'path': str(path), 'split': 'flickr24k', 'sequence': path.stem,
                            'source': 'flickr', 'upscale': True, 'role': 'val' if k % VAL_EVERY == 0 else 'train'})
    sizes = np.array([short_side(path) for path in kept]) if kept else np.zeros(1)
    print(f'  lọc {len(picked)} ảnh trong {time.perf_counter() - started:.0f} s: {dict(reasons)} (ngưỡng nét {cut:.0f}) | '
          f'cạnh ngắn p10/trung vị/p90 {np.percentile(sizes, [10, 50, 90]).round().astype(int).tolist()}, '
          f'{(sizes < SCENE_SHORT).mean():.0%} được phóng lên {SCENE_SHORT}')
else:
    print('BACKGROUND_FRACTION = 0: không dùng ảnh nền Flickr24K')

POOLS = {}
if SCENES:
    POOLS['visdrone'] = SCENES_BY_SPLIT
if BACKGROUNDS:
    POOLS['flickr'] = {role: [s for s in BACKGROUNDS if s['role'] == role] for role in ('train', 'val')}
assert POOLS, 'Không có ảnh sạch nào: gắn VisDrone hoặc để Cell 4b tải Flickr24K (BACKGROUND_FRACTION > 0)'
SHARES = ({'flickr': BACKGROUND_FRACTION, 'visdrone': 1.0 - BACKGROUND_FRACTION} if len(POOLS) == 2
          else {next(iter(POOLS)): 1.0})
SHARES = {source: share for source, share in SHARES.items() if share > 0}


def pick_scene(rng, role):
    """Ảnh sạch ngẫu nhiên cho xem trước: nguồn theo SHARES, rồi một ảnh của ``role`` (không có thì train)."""
    sources = sorted(SHARES)
    source = sources[int(rng.choice(len(sources), p=[SHARES[s] for s in sources]))]
    pool = POOLS[source][role] or POOLS[source]['train']
    return pool[int(rng.integers(len(pool)))]


for source, share in SHARES.items():
    want = round(N_PAIRS * share)
    have = sum(s['name'] not in USED_SCENES for role in ('train', 'val') for s in POOLS[source][role])
    print(f'{source}: {share:.0%} cặp ≈ {want} | ảnh train {len(POOLS[source]["train"])}, valid {len(POOLS[source]["val"])}'
          + (f' | chưa dùng ở bộ cũ: {have}' if source == 'visdrone' and USED_SCENES else '')
          + (f' | ÍT HƠN số cặp: ' + ('ảnh đã dùng ở bộ cũ bù phần thiếu (giảm 1 − BACKGROUND_FRACTION hoặc gắn bản '
                                      'VisDrone gốc)' if source == 'visdrone' and USED_SCENES else
                                      f'mỗi ảnh dùng ~{want / max(1, have):.1f} lần (tăng BACKGROUND_CANDIDATES nếu là Flickr)')
             if want > have else ''))

## Cell 5 — Xem trước và đo độ lóe (chạy lại cell này sau mỗi lần sửa Cell 1)

- `preview_ghep.png`: 20 cặp **ảnh có lóe → ảnh đích**, nhãn ghi hệ số sáng lớp lóe (R, G, B), hệ số sáng cảnh, γ.
- `preview_wu_that.png`: 20 cặp thật của Wu cùng khổ (khi gắn tập test Wu), để so bằng mắt.
- Bảng **độ lóe**: `lóe TB` = ánh sáng lóe cộng thêm trung bình trên khung (tuyến tính, 0–1); `vùng lóe` = tỉ lệ pixel
  sáng thêm > 0,1; `bão hoà` = tỉ lệ pixel cháy trắng ở ảnh vào. Trung vị của mình cao hơn hẳn của Wu thì giảm
  `FLARE_MAX_GAIN`; thấp hơn thì tăng (hoặc chuyển `FLARE_GAIN_MODE = 'shared'` với `FLARE_MIN_GAIN` > 0).

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(SEED)
preview = []
for k in range(N_PREVIEW):
    scene, flare = pick_scene(rng, 'train'), pick_flare(rng, 'train')
    lq, gt, _, params = compose(scene['path'], flare, int(rng.integers(2**31)), scene['upscale'])
    gains = '/'.join(f'{g:.1f}' for g in params['flare_gain'])
    source = 'FlareX' if flare['part'] == 'flarex' else f'Wu {flare["part"][:3]}'
    preview.append((f'{scene["source"][:4]} | {source} {flare["name"][:10]} lóe {gains} | cảnh {params["scene_gain"]:.2f}',
                    lq, gt))
sheet = pair_sheet(preview, OUTPUT_DIR / 'preview_ghep.png',
                   f'Ghép lóe → {"+".join(sorted(SHARES))}: vào → đích | Wu: {FLARE_GAIN_MODE} {FLARE_MIN_GAIN}–{FLARE_MAX_GAIN}, '
                   f'GHOST_BOOST {GHOST_BOOST} | lóe {", ".join(f"{p} {s:.0%}" for p, s in FLARE_USED.items())} | FlareX: '
                   f'{FLAREX_BRIGHTNESS}, cạnh {FLAREX_SIDE} | SCENE_GAIN {SCENE_GAIN}')
plt.figure(figsize=(20, 20 * sheet.height / sheet.width))
plt.imshow(sheet)
plt.axis('off')
plt.show()

if WU_TEST_PAIRS.get('real'):
    real = [(f'Wu real/{name}', np.asarray(Image.open(a).convert('RGB')), np.asarray(Image.open(b).convert('RGB')))
            for name, a, b in WU_TEST_PAIRS['real'][:20]]
    reference = pair_sheet(real, OUTPUT_DIR / 'preview_wu_that.png', 'Wu et al., ảnh thật: vào → đích')
    plt.figure(figsize=(20, 20 * reference.height / reference.width))
    plt.imshow(reference)
    plt.axis('off')
    plt.show()

rows = {}
started = time.perf_counter()
ours = defaultdict(list)
for k in range(N_STATS):
    scene, flare = pick_scene(rng, 'train'), pick_flare(rng, 'train')
    lq, gt, _, _ = compose(scene['path'], flare, int(rng.integers(2**31)), scene['upscale'])
    ours['mình: FlareX' if flare['part'] == 'flarex' else 'mình: Wu'].append(flare_strength(lq, gt))
rows.update(ours)
for split, pairs in WU_TEST_PAIRS.items():
    measured = []
    for _, a, b in pairs:
        lq, gt = np.asarray(Image.open(a).convert('RGB')), np.asarray(Image.open(b).convert('RGB'))
        if lq.shape == gt.shape:
            measured.append(flare_strength(lq, gt))
    rows[f'Wu {split}'] = measured
print(f'\nđộ lóe (p25 / trung vị / p75), {N_STATS} cặp của mình đo trong {time.perf_counter() - started:.0f} s')
print(f'{"":<16}{"n":>5}  {"lóe TB":>22}  {"vùng lóe":>22}  {"bão hoà":>22}')
for label, values in rows.items():
    if not values:
        continue
    values = np.array(values)
    cells = ['  '.join(f'{v:.3f}' for v in np.percentile(values[:, i], [25, 50, 75])) for i in range(3)]
    print(f'{label:<16}{len(values):>5}  ' + '  '.join(f'{c:>22}' for c in cells))

## Cell 6 — Ghép dataset

`N_PAIRS` cặp, chia giữa Flickr và VisDrone theo `BACKGROUND_FRACTION`; trong mỗi nguồn, số cặp valid theo tỉ lệ
ảnh valid (VisDrone test-dev, ảnh Flickr thứ 0, `VAL_EVERY`…), dùng **cả** lớp lóe nhóm valid; cặp train dùng phần
còn lại. Trong mỗi khối (nguồn ảnh nền × split) lớp lóe chia **đúng** theo `FLARE_SHARES`
(phần dư lớn nhất), rồi xáo. Ảnh VisDrone bộ cũ đã dùng xếp sau cùng (chỉ dùng khi hết ảnh chưa dùng). Cuối cell kiểm mọi ảnh đã ghi đúng
`OUT_SIZE`×`OUT_SIZE` và in số cặp còn trùng cảnh với bộ cũ. Ảnh sạch được xoay vòng (mỗi ảnh ~1 lần), lớp lóe rút ngẫu nhiên. Mỗi cặp có seed riêng (`SEED`, số thứ tự),
nên chạy lại ra y hệt. Ghi `input/00000.png`, `gt/00000.png` và `index.csv` (ảnh sạch, lớp lóe, seed, tham số, độ lóe).

In [ ]:
if not BUILD_DATASET:
    print('BUILD_DATASET = False: bỏ qua')
else:
    if UPLOAD.exists():
        shutil.rmtree(UPLOAD)
    for folder in ('input', 'gt'):
        (UPLOAD / folder).mkdir(parents=True)
    rng = np.random.default_rng(SEED + 1)
    PLAN = {source: round(N_PAIRS * share) for source, share in sorted(SHARES.items())}
    PLAN[max(PLAN, key=PLAN.get)] += N_PAIRS - sum(PLAN.values())  # làm tròn: tổng đúng N_PAIRS
    jobs = []
    for source, total_pairs in PLAN.items():
        pools = POOLS[source]
        n_val = round(total_pairs * len(pools['val']) / max(1, len(pools['train']) + len(pools['val'])))
        for split, count in (('train', total_pairs - n_val), ('val', n_val)):
            pool = [pools[split][i] for i in rng.permutation(len(pools[split]))]
            pool.sort(key=lambda scene: scene['name'] in USED_SCENES)  # ổn định: chưa dùng trước, vẫn ngẫu nhiên
            parts = flare_parts(rng, count)
            for k in range(count):
                scene = pool[k % len(pool)]
                jobs.append((len(jobs), scene, pick_flare(rng, split, parts[k]), int(rng.integers(2**31)), split,
                             str(UPLOAD)))
    print('kế hoạch:', {source: dict(Counter(job[4] for job in jobs if job[1]['source'] == source)) for source in PLAN},
          '| lóe:', dict(Counter(job[2]['part'] for job in jobs)))
    started, ROWS, FAILED = time.perf_counter(), [], []
    with ProcessPoolExecutor(max_workers=os.cpu_count() or 4) as pool:
        for done, row in enumerate(pool.map(compose_job, jobs, chunksize=8), 1):
            (ROWS if row['file'] else FAILED).append(row)
            if done % 250 == 0 or done == len(jobs):
                print(f'{done}/{len(jobs)} cặp, {len(FAILED)} lỗi, {time.perf_counter() - started:.0f} s', flush=True)
    for row in FAILED[:10]:
        print('LỖI', row['scene'], row['error'])
    FIELDS = ['file', 'split', 'scene', 'scene_split', 'flare', 'seed', 'flare_mean', 'flare_area', 'saturated', 'params']
    with open(UPLOAD / 'index.csv', 'w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=FIELDS)
        writer.writeheader()
        writer.writerows(ROWS)
    COUNTS = {folder: len(list((UPLOAD / folder).glob('*.png'))) for folder in ('input', 'gt')}
    assert COUNTS['input'] == COUNTS['gt'] == len(ROWS), f'số ảnh lệch: {COUNTS}, {len(ROWS)} hàng'
    SPLIT_COUNTS = Counter(row['split'] for row in ROWS)
    strength = np.array([[row['flare_mean'], row['flare_area'], row['saturated']] for row in ROWS])
    SCENE_COUNTS = Counter('flickr' if row['scene_split'] == 'flickr24k' else 'visdrone' for row in ROWS)
    sizes = Counter()
    for folder in ('input', 'gt'):
        for path in (UPLOAD / folder).iterdir():
            with Image.open(path) as image:  # chỉ đọc phần đầu file
                sizes[image.size] += 1
    assert set(sizes) == {(OUT_SIZE, OUT_SIZE)}, f'cỡ ảnh: {dict(sizes)}'
    REUSED = sum(row['scene'] in USED_SCENES for row in ROWS)
    print(f'cỡ: mọi {sum(sizes.values())} ảnh đều {OUT_SIZE}×{OUT_SIZE} | trùng ảnh nền với bộ cũ: {REUSED} cặp'
          + (f' ({REUSED / len(ROWS):.1%})' if ROWS else '') + ('' if USED_ROOTS else ' (không gắn bộ cũ để đối chiếu)'))
    FLARE_COUNTS = Counter(row['flare'].split('/')[0] for row in ROWS)
    print(f'{len(ROWS)} cặp ({dict(SPLIT_COUNTS)}, ảnh sạch {dict(SCENE_COUNTS)}, lóe '
          + ', '.join(f'{part} {n} ({n / len(ROWS):.0%})' for part, n in FLARE_COUNTS.most_common()) + f') | lóe TB trung vị {np.median(strength[:, 0]):.3f}, '
          f'vùng lóe {np.median(strength[:, 1]):.3f}, bão hoà {np.median(strength[:, 2]):.3f}')

## Cell 7 — README

In [ ]:
if BUILD_DATASET:
    settings = {name: globals()[name] for name in (
        'OUT_SIZE', 'SCENE_SHORT', 'GAMMA', 'FLARE_MAX_GAIN', 'FLARE_GAIN_MODE', 'FLARE_MIN_GAIN', 'SCENE_GAIN', 'NOISE',
        'BLUR_SIGMA', 'DC_OFFSET', 'ROTATION', 'SHIFT_STD', 'SHEAR', 'SCALE', 'WU_SIZE', 'FLARE_SHARES', 'KEEP_LIGHT',
        'FEATHER', 'GHOST_BOOST', 'GHOST_OPEN', 'GHOST_CAP', 'GHOST_PARTS', 'FLAREX_SIDE',
        'FLAREX_BRIGHTNESS', 'FLAREX_MAX_MEAN', 'FLAREX_SCALE', 'FLAREX_TRANSLATE',
        'LIGHT_LOW', 'LIGHT_HIGH', 'SEED', 'N_PAIRS', 'VAL_EVERY', 'BACKGROUND_FRACTION', 'BACKGROUND_CANDIDATES',
        'BACKGROUND_MIN_SHORT', 'BACKGROUND_MIN_LUMA', 'BACKGROUND_MIN_CHROMA', 'BACKGROUND_MAX_SATURATED',
        'BACKGROUND_DROP_BLURRY', 'VISDRONE_AVOID_USED')}
    SOURCE_COUNTS = Counter(row['flare'].split('/')[0] for row in ROWS)
    flarex_note = (f"""

{SOURCE_COUNTS['flarex']} pairs use instead a Flare2D layer of **FlareX** (L. Qu et al., "FlareX: A Physics-Informed
Dataset for Lens Flare Removal via 2D Synthesis and 3D Rendering", NeurIPS 2025; Kaggle `lishenqu/flarex`, license
listed as unknown; code GPL-3.0), resized from 1440 to {FLAREX_SIDE} px in linear light and composited as in FlareX's
`flare7kpp_dataset.py` (random gamma, background removal, affine with the light-source image, brightness
{FLAREX_BRIGHTNESS}, lowered when needed so the mean added flare stays under {FLAREX_MAX_MEAN}, blur; the target is the scene plus FlareX's light-source image).""" if SOURCE_COUNTS['flarex'] else '')
    scene_sources = []
    if SCENE_COUNTS['flickr']:
        scene_sources.append(
            f'Flickr photos from the Flickr24K background set released with Flare7K (Dai et al., NeurIPS 2022; 23,949 '
            f'images sampled from Zhang et al., CVPR 2018, flare-corrupted and overexposed ones filtered out): '
            f'{BACKGROUND_CANDIDATES} drawn at random, kept when the short side is at least {BACKGROUND_MIN_SHORT} px '
            f'(short sides below {SCENE_SHORT} are upscaled to it with Lanczos), the mean luma at least '
            f'{BACKGROUND_MIN_LUMA}, not greyscale, not overexposed, and not among the {BACKGROUND_DROP_BLURRY:.0%} '
            f'blurriest (variance of the Laplacian)')
    if SCENE_COUNTS['visdrone']:
        scene_sources.append(
            'daytime drone traffic photos from VisDrone2019-DET (Zhu et al., TPAMI 2021)'
            + (f', {REUSED} of them also used by visdrone-lens-flare-640 (the others are frames that set did not use, '
               'from the same sequences)' if USED_ROOTS else ''))
    README = f"""# {TITLE}

{len(ROWS)} pairs of {OUT_SIZE}x{OUT_SIZE} images: `input/` has lens flare, `gt/` is the same crop without it.
Clean images (pairs per source: {dict(SCENE_COUNTS)}): {'; and '.join(scene_sources)}. Flare layers: the
flare-only images of Wu et al., "How to Train Neural Networks for Flare Removal", ICCV 2021 (CC BY 4.0,
`gs://gresearch/lens-flare`, resized to 640 px and padded square).

Synthesis follows `flare_removal/python/synthesis.py:add_flare` of the paper's code (random gamma, background
removal, random affine, white balance, random RGB gain, blur, DC offset on the flare; random crop, flips, noise and
digital gain on the scene; flare added in linear light). The captured flares' ghosts and halo rings, 10-40x dimmer than
the core, are brought up: the ghost part (a morphological opening of the flare level, which drops thin streaks, minus the
angular median of the level at each radius around the light-source core, i.e. the round glare) is added
GHOST_BOOST - 1 more times in the flare's own colour (up to GHOST_CAP in linear light), so streaks and the glare around the sun keep their size
(GHOST_BOOST = 1 is the paper's synthesis).{flarex_note} {"The target keeps the light-source core (the near-saturated part of the flare layer, weight rising linearly from 225 to 252)." if KEEP_LIGHT else "The target has no flare and no light source, as in the paper."}

Settings: `{json.dumps(settings)}`

## Files

- `input/NNNNN.png`, `gt/NNNNN.png`: {dict(SPLIT_COUNTS)} pairs. Validation pairs use held-out scenes (VisDrone
  test-dev, no shared sequence with train/val; every {VAL_EVERY}th kept Flickr image by name, `scene_split` flickr24k)
  and held-out flare groups (every {VAL_EVERY}th block of 10 captured frames,
  every {VAL_EVERY}th simulated aperture).
- `index.csv`: `file, split, scene, scene_split, flare, seed, flare_mean, flare_area, saturated, params` (params:
  the random draws of each pair, JSON).

Flare layers per pair: {dict(SOURCE_COUNTS)} (`index.csv` column `flare`).

**Terms.** The Flickr24K photos belong to their owners (research use; cite Flare7K and Zhang et al.); VisDrone
states no license (research use, cite the TPAMI paper); the Wu et al. flares are CC BY 4.0
(cite the ICCV paper); FlareX: cite the NeurIPS paper. This derived set is for research.
"""
    (UPLOAD / 'README.md').write_text(README, encoding='utf-8')
    total = sum(path.stat().st_size for path in UPLOAD.rglob('*') if path.is_file())
    VISIBLE_PUBLIC = PUBLIC and not FAILED
    if UPLOAD_BY_API:
        (UPLOAD / 'dataset-metadata.json').write_text(json.dumps(
            {'title': TITLE, 'id': DATASET, 'licenses': [{'name': 'other'}], 'subtitle': SUBTITLE,
             'description': README}, indent=2, ensure_ascii=False))
    assert 6 <= len(TITLE) <= 50 and 20 <= len(SUBTITLE) <= 80, 'Kaggle: title 6–50 ký tự, subtitle 20–80 ký tự'
    print(f'{UPLOAD}: {len(ROWS)} cặp + index.csv, README.md | {gb(total)} | dataset sẽ',
          'PUBLIC' if VISIBLE_PUBLIC else 'RIÊNG TƯ')

## Cell 8 — Upload

**Có khoá:** `kaggle datasets create -p /tmp/upload/wu_visdrone --dir-mode zip` (đã có thì thêm version mới); cell
đọc cả chữ trong output vì CLI có lúc in lỗi mà vẫn trả mã 0. **Không có khoá:** `input.tar`, `gt.tar` không nén cùng
`index.csv`, `README.md` vào `/kaggle/working`; Commit xong thì tạo dataset từ *Output → New Dataset*.

In [ ]:
import tarfile
import tempfile

PACKED, UPLOADED = {}, False
if not BUILD_DATASET:
    print('BUILD_DATASET = False: bỏ qua')
elif UPLOAD_BY_API:
    free = shutil.disk_usage(tempfile.gettempdir()).free
    assert free > total * 1.05 + 1e9, f'{tempfile.gettempdir()} còn {gb(free)}, zip tạm cần {gb(total)}'


    def dataset_exists():
        code_, text = kaggle('datasets', 'status', DATASET, env=ENV)
        return code_ == 0 and not any(word in text.lower() for word in ('404', 'not found', '403', 'forbidden'))


    EXISTS = dataset_exists()  # hỏi lại: Cell 3 có thể đã cũ (vd chạy Cell 8 lần hai trong cùng phiên)
    if EXISTS:
        args = ['datasets', 'version', '-p', str(UPLOAD), '--dir-mode', 'zip', '-m', f'{len(ROWS)} pairs, seed {SEED}']
    else:
        args = ['datasets', 'create', '-p', str(UPLOAD), '--dir-mode', 'zip'] + (['--public'] if VISIBLE_PUBLIC else [])
    started = time.perf_counter()
    code_, text = kaggle(*args, env=ENV, timeout=4 * 3600)
    print(text[-4000:])
    print(f'{time.perf_counter() - started:.0f} s')
    failed = [word for word in ('error', 'unsuccessful', 'skipping') if word in text.lower()]
    if (code_ != 0 or failed) and not EXISTS and dataset_exists():
        # CLI báo lỗi nhưng dataset đã được tạo (gặp 21/10/2026): Cell 9 kiểm từng thư mục trước khi coi là xong
        print(f'CẢNH BÁO: kaggle CLI trả mã {code_} ({failed}) nhưng {DATASET} đã có; Cell 9 sẽ kiểm số file')
    else:
        assert code_ == 0 and not failed, f'Upload thất bại (mã {code_}, thấy {failed}); xem output ở trên'
    UPLOADED = True
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    free = shutil.disk_usage(OUTPUT_DIR).free
    assert free > total + 1e9 and total < 19e9, f'{OUTPUT_DIR} còn {gb(free)}, cần {gb(total)} (Kaggle lưu tối đa 20 GB)'
    for folder in ('input', 'gt'):
        target, partial = OUTPUT_DIR / f'{folder}.tar', OUTPUT_DIR / f'{folder}.tar.part'
        written = {}
        with tarfile.open(partial, 'w', format=tarfile.PAX_FORMAT) as tar:
            for path in sorted((UPLOAD / folder).iterdir()):
                tar.add(path, arcname=path.name, recursive=False)
                written[path.name] = path.stat().st_size
        with tarfile.open(partial) as tar:
            read_back = {member.name: member.size for member in tar if member.isfile()}
        assert read_back == written, f'{partial}: tar đọc lại khác lúc ghi'
        os.replace(partial, target)
        PACKED[folder] = len(written)
        print(f'{target.name}: {len(written)} file, {gb(target.stat().st_size)}, đọc lại khớp')
    for name in ('index.csv', 'README.md'):
        shutil.copy2(UPLOAD / name, OUTPUT_DIR / name)
    print(f"""
Commit xong: tab Output → New Dataset, tên {SLUG}; chọn input.tar, gt.tar, index.csv, README.md (cùng hai ảnh
preview nếu muốn); đặt {"Public" if VISIBLE_PUBLIC else "Private"}; dán README.md làm mô tả. Kaggle giải nén input.tar
thành input/ ({PACKED.get('input')} file) và gt.tar thành gt/ ({PACKED.get('gt')} file).""")

## Cell 9 — Xác minh trên Kaggle, dọn dẹp

In [ ]:
if BUILD_DATASET and UPLOADED:
    status = ''
    for _ in range(120):  # chờ Kaggle giải nén, tối đa 60 phút
        code_, text = kaggle('datasets', 'status', DATASET, env=ENV)
        status = (text.splitlines() or [''])[-1].strip().lower()
        if status in ('ready', 'error'):
            break
        time.sleep(30)
    REMOTE = remote_files(DATASET, ENV) if status == 'ready' else []
    remote_counts = Counter(posixpath.dirname(row['name']) or '.' for row in REMOTE)
    local_counts = Counter({'input': COUNTS['input'], 'gt': COUNTS['gt'], '.': 2})
    MISMATCHED = [(folder, local_counts[folder], remote_counts[folder])
                  for folder in sorted(set(local_counts) | set(remote_counts)) if local_counts[folder] != remote_counts[folder]]
    if MISMATCHED and REMOTE:
        # API liệt kê file có lúc dừng ở 4 000 file (đo 21/10/2026: 4 002 file, liệt kê ra 4 000) dù file vẫn có:
        # tải thử chính các file "thiếu" (tối đa 6) trước khi coi là lệch.
        listed = {row['name'] for row in REMOTE}
        local_names = [f'{folder}/{name}' for folder in ('input', 'gt') for name in sorted(
            path.name for path in (UPLOAD / folder).iterdir())] + ['index.csv', 'README.md']
        missing = [name for name in local_names if name not in listed]
        probe = Path(tempfile.mkdtemp())
        found = [name for name in missing[:6] if kaggle('datasets', 'download', DATASET, '-f', name, '-p', str(probe),
                                                         '-q', env=ENV, timeout=600)[0] == 0]
        shutil.rmtree(probe, ignore_errors=True)
        print(f'API liệt kê thiếu {len(missing)} file; tải thử {min(6, len(missing))} file: {len(found)} có')
        if len(missing) <= 6 and len(found) == len(missing):
            MISMATCHED = []
    print(f'https://www.kaggle.com/datasets/{DATASET} | trạng thái {status} | Kaggle {len(REMOTE)} file | '
          + ('khớp từng thư mục' if REMOTE and not MISMATCHED else f'LỆCH {MISMATCHED}' if REMOTE else 'chưa liệt kê được'))
    REPORT.write_text(f'# {DATASET}\n\n- trạng thái {status}\n- {dict(SPLIT_COUNTS)}\n- lệch: {MISMATCHED or "không"}\n',
                      encoding='utf-8')
    assert REMOTE and not MISMATCHED, 'dataset trên Kaggle chưa khớp; xem dòng trên'
shutil.rmtree(UPLOAD.parent, ignore_errors=True)
shutil.rmtree(BACKGROUND_DIR, ignore_errors=True)  # zip giữ lại: chạy lại notebook trong phiên không tải lại
ENV = None  # khoá chỉ từng nằm trong biến này
print('xong; /kaggle/working:', sorted(path.name for path in OUTPUT_DIR.iterdir()) if OUTPUT_DIR.exists() else [])